In [1]:
import pandas as pd
import numpy as np

In [2]:
dl_data = pd.read_excel('5G_DL.xlsx', sheet_name='Series Formatted Data')
ul_data = pd.read_excel('5G_UL.xlsx', sheet_name='Series Formatted Data')
scanner_data = pd.read_excel('5G_Scanner.xlsx', sheet_name='Series Formatted Data')

dl_data = dl_data.drop(["Message","Technology_Mode"],axis=1)
dl_data = dl_data.loc[:, ~dl_data.columns.str.contains('^Unnamed')]

ul_data = ul_data.drop(["Message","Technology_Mode"],axis=1)
ul_data = ul_data.loc[:, ~ul_data.columns.str.contains('^Unnamed')]

scanner_data = scanner_data.drop(["Message"],axis=1)
scanner_data = scanner_data.loc[:, ~scanner_data.columns.str.contains('^Unnamed')]

In [19]:
def group_by_time(data):
    new_data = pd.DataFrame()

    last_known_data_cols = data.select_dtypes(include=['object']).columns.tolist()
    mean_cols = data.select_dtypes(include=['float',"int"]).columns.tolist()

    for row in data['Time'].unique():

        filtered_row_mean = data[data["Time"] == row][mean_cols].mean()
        filtered_row_last = data[data["Time"] == row][last_known_data_cols]
        last_known_values = filtered_row_last.apply(lambda x: x.dropna().iloc[-1] if not x.dropna().empty else np.nan)
        filtered_row = pd.concat([pd.Series({'Time': row}), filtered_row_mean, last_known_values])
        new_data = pd.concat([new_data, filtered_row.to_frame().T], ignore_index=True)
                
    return new_data

In [ ]:
dl_data = group_by_time(dl_data)
ul_data = group_by_time(ul_data)
scanner_data = group_by_time(scanner_data)

dl_data["Time"] = pd.to_datetime(dl_data["Time"])
ul_data["Time"] = pd.to_datetime(ul_data["Time"])
scanner_data["Time"] = pd.to_datetime(scanner_data["Time"])

dl_data.columns = ["Time" if x== "Time" else "Longitude" if x=="Longitude" else "Latitude" if x=="Latitude" else f"DL_{x}" for x in dl_data.columns]
ul_data.columns = ["Time" if x== "Time" else "Longitude" if x=="Longitude" else "Latitude" if x=="Latitude" else f"UL_{x}" for x in ul_data.columns]
scanner_data.columns = ["Time" if x== "Time" else "Longitude" if x=="Longitude" else "Latitude" if x=="Latitude" else f"SC_{x}" for x in scanner_data.columns]

joined_time_set = set([*dl_data["Time"].unique(),*ul_data["Time"].unique(),*scanner_data["Time"].unique()])
df = pd.DataFrame()

df["Time"] = sorted(list(joined_time_set))
df = df.merge(dl_data, how="left", on="Time")
df = df.merge(ul_data, how="left", on="Time")
df = df.merge(scanner_data, how="left", on="Time")

for col in ["Latitude", "Longitude"]:
    df[col] = (
        df.get(col)
        .combine_first(df.get(f"{col}_x"))
        .combine_first(df.get(f"{col}_y"))
    )
    df = df.drop(columns=[f"{col}_x",f"{col}_y"])


In [48]:
df

,Time,DL_NR_UE_PCI_0,DL_NR_UE_RSRP_0,DL_NR_UE_RSRQ_0,DL_NR_UE_SINR_0,DL_NR_UE_Nbr_PCI_0,DL_NR_UE_Nbr_PCI_1,DL_NR_UE_Nbr_PCI_2,DL_NR_UE_Nbr_PCI_3,DL_NR_UE_Nbr_PCI_4,...,SC_NR_Scan_SSB_RSRQ_SortedBy_RSRP_4,SC_NR_Scan_SSB_RSRQ_SortedBy_RSRP_5,SC_NR_Scan_SSB_RSRQ_SortedBy_RSRP_6,SC_NR_Scan_SSB_SINR_SortedBy_RSRP_0,SC_NR_Scan_SSB_SINR_SortedBy_RSRP_1,SC_NR_Scan_SSB_SINR_SortedBy_RSRP_2,SC_NR_Scan_SSB_SINR_SortedBy_RSRP_3,SC_NR_Scan_SSB_SINR_SortedBy_RSRP_4,SC_NR_Scan_SSB_SINR_SortedBy_RSRP_5,SC_NR_Scan_SSB_SINR_SortedBy_RSRP_6
0,2025-03-14 12:14:32.955,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-03-14 12:14:32.966,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025-03-14 12:14:33.127,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2025-03-14 12:14:33.135,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2025-03-14 12:14:33.444,48.0,-87.1,-11.2,7.3,76.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21045,2025-03-14 12:32:17.647,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
21046,2025-03-14 12:32:17.649,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
21047,2025-03-14 12:32:17.650,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
21048,2025-03-14 12:32:17.781,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
